# 삼양엔씨켐 매출 × 경제지표 상관관계 분석

삼양엔씨켐은 반도체 노광/세정 공정용 정밀화학 소재(포토레지스트 원료, 세정액)를 만드는
비상장 계열사다. **`samyang_food_correlation.ipynb`/`samyang_packaging_correlation.ipynb`와
달리 분기·반기보고서가 없다** — 2020~2023년은 감사보고서만, 2024년부터 사업보고서를 내기
시작했다. 그래서 분기 단위가 아니라 **연 단위**로 분석한다 (표본 5개 연도).

> **표본 크기 주의**: n=5로 매우 작다. 여기서 나온 상관계수는 참고용 신호조차 되기 어렵고,
> "방향성만 보는" 수준으로 해석해야 한다. 통계적 유의성을 주장할 수 있는 표본이 아니다.

> **지표 한계**: 반도체 소재 산업에 직접 대응하는 지표(예: 메모리 반도체 가격, D램 현물가)가
> 현재 DB에 없다. 사업보고서에 명시된 환위험 노출(수출 비중 높음, USD/JPY 결제)에 착안해
> 환율·수입물가 위주로 대체 지표를 썼다 — 업종 특성을 온전히 반영하지는 못한다.

공통 계산/시각화 함수는 `eda_utils.py`(같은 폴더)에 있다.

## 0. 환경 설정

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

sys.path.append(str(Path("../../RAG").resolve()))  # dart_parser.py가 있는 폴더
import dart_parser
import eda_utils  # 계열사 노트북 공통 로직 (같은 mandu/Eda 폴더)

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.family"] = "Malgun Gothic"  # Windows 기본 한글 폰트 (다른 OS라면 폰트명 교체)

In [ ]:
DART_DIR = Path("../../RAG/data/dart_xml")
TARGET_COMPANIES = {"(주)엔씨켐", "(주)삼양엔씨켐"}  # 2024년 사명 변경 전/후 표기

# 지표 DB(DATABASE_URL)는 Steam_Sales/dashboard/backend/.env 에 있다
DASHBOARD_ENV = Path("../../dashboard/backend/.env")
load_dotenv(DASHBOARD_ENV)

## 1. DART 공시에서 연간 매출액 추출

분기가 없으니 누적치를 분기로 역산하는 과정이 필요 없다. 감사보고서/사업보고서 1건당
연간 매출액 1건을 그대로 쓴다 (`eda_utils.extract_annual_metric`가 보고서 종류별로
다른 표 형식을 자동으로 처리한다).

In [ ]:
annual_df, failed = eda_utils.extract_annual_metric(DART_DIR, TARGET_COMPANIES, value_col="revenue")
annual_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(annual_df.index, annual_df["revenue"], marker="o", color="#8A4FBE")
ax.fill_between(annual_df.index, annual_df["revenue"], annual_df["revenue"].min() * 0.95, alpha=0.12, color="#8A4FBE")
ax.set_title("Samyang NCChem - Annual Revenue (KRW million)")
ax.set_ylabel("KRW million")
ax.grid(alpha=0.3)
fig.autofmt_xdate()
plt.show()

## 2. DB에서 경제지표 로드

사업보고서에 명시된 환위험(USD/JPY 결제, 수출 비중 높음) 노출에 맞춰 환율·수입물가
위주로 골랐다. 반도체 업황을 직접 반영하는 지표는 DB에 없다.

In [ ]:
from sqlalchemy import create_engine
import os

engine = create_engine(os.environ["DATABASE_URL"], pool_pre_ping=True)

TARGETS = {
    "market_yfinance": ["원달러환율", "달러인덱스", "미국국채10년", "KOSPI"],
    "import_price_index": ["한국", "미국"],
    "samyang_stock_prices": ["삼양엔씨켐"],  # 2025년 상장이라 그 이전 구간은 값이 비어있음
}

all_series = eda_utils.load_indicator_series(engine, TARGETS)
print(f"{len(all_series)}개 지표 시계열 로드")

## 3. 연도별 지표 평균과 매출 정렬

In [ ]:
aligned = eda_utils.align_indicators_to_periods(annual_df, "revenue", all_series)
aligned

## 4. 레벨 기준 상관관계

표본이 5개뿐이라 `min_n`을 낮춰서 계산한다 (기본값 6이면 전부 걸러짐).

In [ ]:
level_corr = eda_utils.corr_table(aligned, all_series, "revenue", min_n=4)
level_corr

In [ ]:
eda_utils.plot_top_correlations(level_corr, "Level correlation with NCChem revenue (n=5, 참고용)")

## 5. 전기 대비 변화율(%) 기준 상관관계

In [ ]:
pct = aligned.drop(columns=["period_from"]).pct_change().dropna(how="all")
pct_corr = eda_utils.corr_table(pct, all_series, "revenue", min_n=3)
pct_corr

## 6. 시차(Lag) 분석

분기가 아니라 연 단위이므로 lag는 0~1년만 본다 (표본이 5개뿐이라 2년 이상 밀면 비교
가능한 쌍이 3개 미만으로 줄어든다).

> **표본 크기 재차 주의**: 이 분석 전체가 n=5 수준이라, 여기 나온 어떤 상관계수도
> 확정된 관계로 취급하면 안 된다. 상장사(삼양사/삼양패키징) 분석과 같은 신뢰도로
> 보지 말 것.

In [ ]:
level_df = aligned.drop(columns=["period_from"])
lag_df = eda_utils.lag_correlation_table(level_df, all_series, "revenue", max_lag=2, min_n=3)
lag_df

In [ ]:
eda_utils.plot_lag_heatmap(lag_df, "Lag correlation (indicator at t-L year vs revenue at t)", max_lag=2, top_n=len(lag_df))

## 결론 및 한계

*(노트북을 실행한 뒤, 위 상관관계 표를 보고 이 셀에 실제 결론을 채워 넣을 것)*

- 표본이 5개 연도뿐이라 상관계수는 통계적으로 의미 있는 수준이 아니다 — 방향성 참고용.
- 반도체 소재 업황을 직접 반영하는 지표가 DB에 없어 환율·수입물가로 대체했다. 실제
  매출 동인(메모리 반도체 수요, 고객사 재고 사이클 등)을 반영하지 못할 가능성이 크다.
- 2021년 매출이 전년 대비 급증했는데, 이건 이 시기 반도체 업황 호황과 관련 있어 보이나
  이 노트북의 지표로는 설명되지 않는다 — 별도 정성 검토가 필요.